# Building a RAG System — Part 1: Ingestion Pipeline

**Internal Ship Program · Tasks 2.1 → 2.4**

Welcome! This notebook teaches the **ingestion** half of a Retrieval-Augmented Generation (RAG)
system, hands-on, using the **LangChain ecosystem** and **local open-source models** (no API keys,
your data never leaves the machine).

We process a real document — the **CIS Controls v8** security guidelines (`data/`) — through four stages:

```
        ┌──────────┐    ┌──────────┐    ┌───────────┐    ┌──────────────┐
  PDF → │ 2.1 PARSE│ →  │2.2 CHUNK │ →  │2.3 EMBED  │ →  │2.4 STORE     │ → (later: retrieve → rerank → agent)
        │unstructured   │splitters │    │bge-small  │    │Weaviate      │
        └──────────┘    └──────────┘    └───────────┘    └──────────────┘
```

| Stage | What it does | Tool |
|-------|--------------|------|
| **2.1 Parse** | Turn a messy PDF into clean text + structure (titles, tables, images) | `langchain-unstructured` |
| **2.2 Chunk** | Split text into retrieval-sized pieces | `langchain-text-splitters` + unstructured |
| **2.3 Embed** | Turn each chunk into a vector (numbers that capture meaning) | `langchain-huggingface` (`bge-small-en-v1.5`) |
| **2.4 Store** | Save vectors in a database we can search by similarity | `langchain-weaviate` |

> **Why RAG?** LLMs don't know your private docs. RAG lets the model *retrieve* relevant chunks
> from your documents at question time and answer grounded in them. Everything here is the
> "prepare the knowledge" half — retrieval, reranking, and the agent come next"


## 0. Setup

### Python dependencies
Already declared in `pyproject.toml`. If you cloned fresh, run **`uv sync`** in a terminal, then
select this project's `.venv` as the notebook kernel. (Or run the cell below to install on the fly.)

### System packages (required for hi-res PDF parsing)
`unstructured` uses OCR + a layout model to detect tables and images. On Debian/Ubuntu/WSL:

```bash
sudo apt-get update && sudo apt-get install -y poppler-utils tesseract-ocr libgl1
```

- **poppler-utils** → renders PDF pages to images
- **tesseract-ocr** → reads text from those images (OCR)
- **libgl1** → shared library the layout/vision model needs

> First run downloads the embedding model (~130 MB) and the layout model. Be patient once; they're cached after.
